# Week 3 — Data & File Handling

Covers: `pathlib`/`os` directory walking, `re` for text cleaning/chunking, and reading
structured formats (CSV via `pandas`, plus a note on PDF/DOCX extraction).

## 1. `pathlib` and `os` — walking a document set

In [ ]:
import tempfile, os
from pathlib import Path

# Build a small fake "document set" to walk, like a RAG ingestion folder
base = Path(tempfile.mkdtemp())
(base / "policies").mkdir()
(base / "circulars").mkdir()

(base / "policies" / "motor_policy.txt").write_text("Motor policy wording. Exclusion clause 4.2 applies to...")
(base / "circulars" / "irdai_2025_07.txt").write_text("IRDAI circular: settlement timelines must not exceed 30 days.")

print("Walking:", base)
for path in base.rglob("*.txt"):
    print(" -", path.relative_to(base), f"({path.stat().st_size} bytes)")

## 2. Regular expressions — cleaning and chunking text

In [ ]:
import re

raw_text = '''
  Motor Policy Wording   -- Section 4: EXCLUSIONS
  4.1 Wear and tear is excluded.
  4.2   Pre-existing damage is excluded, see Annex A.
'''

# Collapse repeated whitespace — a very common RAG pre-processing step
cleaned = re.sub(r"[ \t]+", " ", raw_text).strip()
print(repr(cleaned))

# Split into clause-like chunks using a numbering pattern (e.g. "4.1", "4.2")
chunks = re.split(r"(?=\d\.\d+\s)", cleaned)
chunks = [c.strip() for c in chunks if c.strip()]
for c in chunks:
    print("CHUNK:", c)

## 3. Reading structured formats — CSV via `pandas`

In [ ]:
import pandas as pd
import io

csv_text = '''claim_id,policy_id,status,amount
C-1001,P-55,approved,15000
C-1002,P-12,rejected,0
C-1003,P-55,approved,8000
'''

claims_df = pd.read_csv(io.StringIO(csv_text))
print(claims_df)
print()
print("Approved total:", claims_df.loc[claims_df.status == "approved", "amount"].sum())

## 4. PDF/DOCX extraction — reference note

This notebook doesn't extract a real PDF/DOCX (those libraries aren't part of the core
runtime and need installing — see `requirements.txt`). The pattern, once installed, is:

```python
# PDF, using pypdf
from pypdf import PdfReader
reader = PdfReader("policy_wording.pdf")
text = "\n".join(page.extract_text() for page in reader.pages)

# DOCX, using python-docx
from docx import Document
doc = Document("policy_wording.docx")
text = "\n".join(p.text for p in doc.paragraphs)
```

Both give you back plain text, which then goes through the same cleaning/chunking pattern
shown in Section 2 above — the library changes, the downstream pipeline doesn't.